# AlphaCluster — Fine-tune v3 (Phase 3 only, multi-asset)

Continue training the v3 model (500k steps) for an additional 2M steps
using **Phase 3 only** (low entropy, full churn/fee penalties) on **multi-asset data**.

**Why Phase 3 only?** v3 already learned to trade patiently (9 trades/ep).
Restarting curriculum from Phase 1 destroys this behavior by encouraging
exploration and reducing penalties. Phase 3 keeps what works and refines it.

**Why multi-asset?** Training on 20 crypto pairs instead of just BTC prevents
overfitting to one market's patterns. The model sees normalized indicators
(returns, RSI, MACD, BB, ATR) — these are asset-agnostic.

**Phase 3 settings:**
- `ent_coef = 0.005` (minimal exploration)
- `fee_scale = 2.0x` (strong fee awareness)
- `churn_penalty_scale = 2.0x` (punish rapid trading)
- `drawdown_penalty_scale = 1.5x`

## Setup

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

!pip install -q "stable-baselines3>=2.4,<3.0" "gymnasium>=1.0,<2.0" pyarrow python-dotenv tqdm rich

import sys
from pathlib import Path

# Add the project source to the Python path
SRC_DIR = "/kaggle/input/alphacluster-repo/src"
sys.path.insert(0, SRC_DIR)

import alphacluster
print(f"AlphaCluster loaded from {Path(alphacluster.__file__).parent}")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected.")

## Load & Split Multi-Asset Data

Load all available `*_5m.parquet` files. Each asset is split chronologically
(70/15/15) and the training sets are passed as a list to `TradingEnv(dfs=...)`.

In [ ]:
import pandas as pd

DATA_DIR = Path("/kaggle/input/alphacluster-data")

# Load all 5m parquet files
parquet_files = sorted(DATA_DIR.glob("*_5m.parquet"))
assert len(parquet_files) > 0, f"No parquet files found in {DATA_DIR}"

all_dfs = {}
for f in parquet_files:
    symbol = f.stem.replace("_5m", "").upper()
    df = pd.read_parquet(f, engine="pyarrow")
    all_dfs[symbol] = df
    date_range = f"{df['open_time'].iloc[0]} to {df['open_time'].iloc[-1]}"
    print(f"  {symbol}: {len(df):>10,} candles  ({date_range})")

print(f"\nLoaded {len(all_dfs)} assets, {sum(len(d) for d in all_dfs.values()):,} total candles")

# Chronological split per asset: 70% train / 15% val / 15% test
train_dfs = []
val_dfs = []
test_dfs = []

for symbol, df in all_dfs.items():
    n = len(df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    train_dfs.append(df.iloc[:train_end].reset_index(drop=True))
    val_dfs.append(df.iloc[train_end:val_end].reset_index(drop=True))
    test_dfs.append(df.iloc[val_end:].reset_index(drop=True))

total_train = sum(len(d) for d in train_dfs)
total_val = sum(len(d) for d in val_dfs)
total_test = sum(len(d) for d in test_dfs)
print(f"Split: train={total_train:,}  val={total_val:,}  test={total_test:,}")

# BTC-only splits for evaluation (consistent with v3 baseline)
btc_df = all_dfs["BTCUSDT"]
n_btc = len(btc_df)
btc_val_df = btc_df.iloc[int(n_btc * 0.70):int(n_btc * 0.85)].reset_index(drop=True)
btc_test_df = btc_df.iloc[int(n_btc * 0.85):].reset_index(drop=True)
print(f"BTC eval: val={len(btc_val_df):,}  test={len(btc_test_df):,}")

## Fine-tune v3 (Phase 3 only, 2M steps, multi-asset)

Load the v3 checkpoint and continue training with `phase1_end=0.0, phase2_end=0.0`
so CurriculumCallback immediately enters Phase 3. Training environments use
`dfs=train_dfs` — each episode randomly samples from all 20 assets.

In [ ]:
import time

import torch
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize

from alphacluster.agent.config import TrainingConfig
from alphacluster.agent.trainer import (
    CurriculumCallback,
    TrainingMetricsCallback,
    create_agent,
    save_agent,
)
from alphacluster.config import MODEL_VERSION
from alphacluster.env.trading_env import TradingEnv

# ── Paths ──
V3_CHECKPOINT = Path("/kaggle/input/alphacluster-models/ppo_trading_simple_final_v3.pt")
MODELS_DIR = Path("/kaggle/working/models")
CHECKPOINT_DIR = MODELS_DIR / "checkpoints"

assert V3_CHECKPOINT.exists(), f"v3 checkpoint not found: {V3_CHECKPOINT}"

FINETUNE_TIMESTEPS = 2_000_000
N_ENVS = 4


class ProgressCallback(BaseCallback):
    """Print a one-line progress update every log_interval timesteps."""

    def __init__(self, total_timesteps: int, log_interval: int = 50_000):
        super().__init__(verbose=0)
        self.total_timesteps = total_timesteps
        self.log_interval = log_interval
        self._start_time = None
        self._next_log = log_interval

    def _on_training_start(self) -> None:
        self._start_time = time.time()

    def _on_step(self) -> bool:
        if self.num_timesteps >= self._next_log:
            elapsed = time.time() - self._start_time
            pct = self.num_timesteps / self.total_timesteps * 100
            fps = self.num_timesteps / max(elapsed, 1)
            eta = (self.total_timesteps - self.num_timesteps) / max(fps, 1)
            print(
                f"  [{pct:5.1f}%] {self.num_timesteps:>10,} / {self.total_timesteps:,} "
                f"| {fps:.0f} fps | ETA {eta/60:.0f}m",
                flush=True,
            )
            self._next_log += self.log_interval
        return True


def make_env(dfs_list, config, rank=0):
    """Factory for SubprocVecEnv with multi-asset support."""
    def _init():
        import os
        import warnings

        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
        warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

        env = TradingEnv(
            dfs=dfs_list,
            window_size=config.window_size,
            episode_length=config.episode_length,
            simple_actions=True,
            fixed_size_pct=config.fixed_size_pct,
            fixed_leverage=config.fixed_leverage,
        )
        env.reset(seed=rank)
        return env
    return _init


# ── Config: Phase 3 only ──
config = TrainingConfig(
    total_timesteps=FINETUNE_TIMESTEPS,
    simple_actions=True,
    fixed_size_pct=0.10,
    fixed_leverage=10,
    eval_freq=100_000,
    phase1_end=0.0,   # Skip Phase 1 entirely
    phase2_end=0.0,   # Skip Phase 2 entirely
)

# ── Environments ──
print(f"Creating {N_ENVS} parallel training environments ({len(train_dfs)} assets) ...")
envs = SubprocVecEnv([make_env(train_dfs, config, rank=i) for i in range(N_ENVS)])
train_env = VecNormalize(envs, norm_obs=False, norm_reward=True, clip_reward=10.0)

# Eval on BTC validation set (consistent with v3 baseline)
eval_env = TradingEnv(
    df=btc_val_df,
    window_size=config.window_size,
    episode_length=config.episode_length,
    simple_actions=True,
    fixed_size_pct=config.fixed_size_pct,
    fixed_leverage=config.fixed_leverage,
)

# ── Load v3 weights ──
print("Creating PPO agent and loading v3 checkpoint ...")
agent = create_agent(train_env, config, verbose=0)
state_dict = torch.load(str(V3_CHECKPOINT), map_location="cpu", weights_only=True)
agent.policy.load_state_dict(state_dict)
print(f"Loaded v3 weights from {V3_CHECKPOINT}")

# ── Callbacks ──
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
curriculum_cb = CurriculumCallback(config, verbose=1)
progress_cb = ProgressCallback(FINETUNE_TIMESTEPS, log_interval=50_000)
metrics_cb = TrainingMetricsCallback(
    eval_env=eval_env,
    log_freq=100_000,
    log_path=CHECKPOINT_DIR / "training_metrics.csv",
    verbose=0,
)

# ── Train ──
print(f"\nFine-tuning v3 for {FINETUNE_TIMESTEPS:,} steps (Phase 3 only, multi-asset)")
print(f"Model version: {MODEL_VERSION}")
print(f"Phase 3: ent_coef=0.005, fee=2.0x, churn=2.0x, drawdown=1.5x")
print(f"Config: size_pct={config.fixed_size_pct}, leverage={config.fixed_leverage}x")
print(f"Assets: {len(train_dfs)} (random per episode)\n")

start = time.time()
agent.learn(
    total_timesteps=FINETUNE_TIMESTEPS,
    callback=CallbackList([curriculum_cb, progress_cb, metrics_cb]),
    progress_bar=False,
)
elapsed = time.time() - start
print(f"\nTraining complete in {elapsed/60:.1f} minutes")

# ── Save ──
MODELS_DIR.mkdir(parents=True, exist_ok=True)
final_path = MODELS_DIR / "ppo_trading_simple_v3_finetuned"
save_agent(agent, final_path)
print(f"Model saved to {final_path}.pt")

train_env.close()

## Evaluation on Test Set

In [ ]:
from alphacluster.backtest.runner import run_backtest
from alphacluster.backtest.metrics import calculate_metrics, print_report

EVAL_SEED = 42
N_EVAL_EPISODES = 5

test_env = TradingEnv(
    df=btc_test_df,
    window_size=config.window_size,
    episode_length=config.episode_length,
    simple_actions=True,
    fixed_size_pct=config.fixed_size_pct,
    fixed_leverage=config.fixed_leverage,
)

print(f"Running backtest on BTC TEST set ({N_EVAL_EPISODES} episodes, seed={EVAL_SEED}) ...")
result = run_backtest(model=agent, env=test_env, n_episodes=N_EVAL_EPISODES, seed=EVAL_SEED)
metrics = calculate_metrics(result)
print_report(metrics)

# ── Compare with v3 baseline ──
print("\n" + "=" * 60)
print("  v3 BASELINE (BTC only) vs FINE-TUNED (multi-asset)")
print("=" * 60)
print(f"  v3 baseline:  -0.95% avg PnL/ep, 8.8 trades/ep, WR=40.9%")
print(f"  Fine-tuned:   {metrics['avg_episode_return_pct']:.2f}% avg PnL/ep, "
      f"{metrics['avg_trades_per_episode']:.1f} trades/ep, "
      f"WR={metrics['win_rate_pct']:.1f}%")
print("=" * 60)

## Learning Curves

In [ ]:
import matplotlib.pyplot as plt

metrics_path = CHECKPOINT_DIR / "training_metrics.csv"
if metrics_path.exists():
    df_metrics = pd.read_csv(metrics_path)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("v3 Fine-tuning Progress (Phase 3 only)", fontsize=14)

    plots = [
        ("timestep", "mean_reward", "Mean Episode Return (%)"),
        ("timestep", "trades_per_episode", "Trades per Episode"),
        ("timestep", "win_rate", "Win Rate (%)"),
        ("timestep", "avg_trade_duration", "Avg Trade Duration (steps)"),
        ("timestep", "flat_pct", "Flat Time (%)"),
        ("timestep", "total_pnl_pct", "Total PnL (%)"),
    ]

    for ax, (x, y, title) in zip(axes.flat, plots):
        if y in df_metrics.columns:
            ax.plot(df_metrics[x], df_metrics[y], marker="o", markersize=3)
        ax.set_title(title)
        ax.set_xlabel("Timesteps")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(df_metrics.to_string(index=False))
else:
    print("No training metrics found.")

## Download

In [ ]:
import os

print("=" * 60)
print("  FINE-TUNING COMPLETE")
print("=" * 60)
print()
print("Model files in /kaggle/working/ (download from Output tab):")
print()

for root, dirs, files in os.walk("/kaggle/working/models"):
    level = root.replace("/kaggle/working/", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024 * 1024)
        print(f"{sub_indent}{f}  ({size_mb:.1f} MB)")

print()
print("To evaluate locally:")
print("  python scripts/evaluate.py --model-path models/ppo_trading_simple_v3_finetuned.pt --simple-actions")